# Лабораторная 2. Формирование отчётов в Apache Spark

Выполнила: Пироженкова Ирина 6403

**Задача:** построить отчёт по 10 наиболее популярным языкам программирования за каждый год с 2010 по 2020, используя posts_sample.xml и programming-languages.csv, а затем сохранить результат в формате Apache Parquet.





## 1. Установка PySpark


In [54]:

!pip -q install pyspark==3.5.1



## 2. Импорты и пути к файлам



In [55]:
import os
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

BASE_DIR = "/content"
POSTS_XML = os.path.join(BASE_DIR, "posts_sample.xml")
LANGUAGES_CSV = os.path.join(BASE_DIR, "programming-languages.csv")
PARQUET_OUTPUT = "/content/top_languages_2010_2020_parquet"

print("POSTS_XML:", POSTS_XML)
print("LANGUAGES_CSV:", LANGUAGES_CSV)
print("PARQUET_OUTPUT:", PARQUET_OUTPUT)

POSTS_XML: /content/posts_sample.xml
LANGUAGES_CSV: /content/programming-languages.csv
PARQUET_OUTPUT: /content/top_languages_2010_2020_parquet



### Проверка, что входные файлы доступны


In [56]:

for path in [POSTS_XML, LANGUAGES_CSV]:
    print(path, "->", os.path.exists(path))


/content/posts_sample.xml -> True
/content/programming-languages.csv -> True


In [57]:
count_2020 = !grep -c 'CreationDate="2020-' /content/posts_sample.xml
count_2020 = int(count_2020[0])

if count_2020 == 0:
    print("В posts_sample.xml нет записей за 2020 год")
else:
    print(f"В posts_sample.xml есть записи за 2020 год: {count_2020}")

В posts_sample.xml нет записей за 2020 год


В используемом файле posts_sample.xml отсутствуют записи за 2020 год, что подтверждено прямой проверкой по полю CreationDate. Поэтому итоговый отчёт фактически строится для 2010–2019, хотя фильтрация в коде задана для диапазона 2010–2020.


## 3. Создание SparkSession


In [58]:

spark = (
    SparkSession.builder
    .appName("lab2_language_report")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(spark.version)


3.5.1



## 4. Подготовка справочника языков

Берём первый столбец из CSV, приводим названия к нижнему регистру и убираем дубликаты.


In [59]:

language_source = spark.read.option("header", True).csv(LANGUAGES_CSV)
name_column = language_source.columns[0]

language_reference = (
    language_source
    .select(F.trim(F.lower(F.col(name_column))).alias("language_name"))
    .where(F.col("language_name").isNotNull())
    .where(F.col("language_name") != "")
    .dropDuplicates()
)

print("Количество языков в справочнике:", language_reference.count())
language_reference.orderBy("language_name").show(10, truncate=False)


Количество языков в справочнике: 698
+-------------+
|language_name|
+-------------+
|@formula     |
|a# (axiom)   |
|a# .net      |
|a+           |
|a++          |
|a-0 system   |
|abap         |
|abc          |
|abc algol    |
|abset        |
+-------------+
only showing top 10 rows




## 5. Чтение XML как текстового файла

В `posts_sample.xml` каждая строка с постом содержит XML-подобную запись. Мы берём только те строки, где есть:
- `CreationDate`
- `Tags`

После этого извлекаем год и строку с тегами регулярными выражениями.


In [60]:

raw_rows = spark.read.text(POSTS_XML).withColumnRenamed("value", "raw_xml")

candidate_posts = (
    raw_rows
    .where(F.col("raw_xml").contains('CreationDate="'))
    .where(F.col("raw_xml").contains('Tags="'))
)

posts_with_year_and_tags = (
    candidate_posts
    .select(
        F.regexp_extract("raw_xml", r'CreationDate="(\d{4})', 1).cast("int").alias("year"),
        F.regexp_extract("raw_xml", r'Tags="([^"]+)"', 1).alias("tag_sequence")
    )
    .where((F.col("year") >= 2010) & (F.col("year") <= 2020))
)

print("Количество постов после фильтрации:", posts_with_year_and_tags.count())
posts_with_year_and_tags.show(5, truncate=False)


Количество постов после фильтрации: 17642
+----+--------------------------------------------------------------+
|year|tag_sequence                                                  |
+----+--------------------------------------------------------------+
|2010|&lt;c++&gt;&lt;character-encoding&gt;                         |
|2010|&lt;sharepoint&gt;&lt;infopath&gt;                            |
|2010|&lt;iphone&gt;&lt;app-store&gt;&lt;in-app-purchase&gt;        |
|2010|&lt;symfony1&gt;&lt;schema&gt;&lt;doctrine&gt;&lt;fixtures&gt;|
|2010|&lt;java&gt;                                                  |
+----+--------------------------------------------------------------+
only showing top 5 rows




## 6. Нормализация тегов

Строка тегов в выборке выглядит примерно так:

`&lt;python&gt;&lt;pandas&gt;&lt;dataframe&gt;`

Мы превращаем её в массив тегов и затем делаем `explode`, чтобы получить одну строку на один тег.


In [61]:

normalized_tags = (
    posts_with_year_and_tags
    .withColumn("tag_sequence", F.regexp_replace("tag_sequence", "&lt;", ""))
    .withColumn("tag_sequence", F.regexp_replace("tag_sequence", "&gt;", " "))
    .withColumn("single_tag", F.explode(F.split(F.trim(F.col("tag_sequence")), r"\s+")))
    .select(
        "year",
        F.trim(F.lower(F.col("single_tag"))).alias("language_name")
    )
    .where(F.col("language_name") != "")
)

normalized_tags.show(10, truncate=False)


+----+------------------+
|year|language_name     |
+----+------------------+
|2010|c++               |
|2010|character-encoding|
|2010|sharepoint        |
|2010|infopath          |
|2010|iphone            |
|2010|app-store         |
|2010|in-app-purchase   |
|2010|symfony1          |
|2010|schema            |
|2010|doctrine          |
+----+------------------+
only showing top 10 rows




## 7. Фильтрация только по языкам программирования

Здесь используется **inner join** со справочником языков. Это отличается от варианта с `broadcast set` и даёт аккуратную Spark-реализацию через DataFrame/SQL.


In [62]:

normalized_tags.createOrReplaceTempView("tag_mentions")
language_reference.createOrReplaceTempView("language_reference")

language_mentions = spark.sql("""
    SELECT
        m.year,
        m.language_name,
        COUNT(*) AS mentions
    FROM tag_mentions m
    INNER JOIN language_reference l
        ON m.language_name = l.language_name
    GROUP BY m.year, m.language_name
""")

language_mentions.orderBy("year", F.desc("mentions")).show(20, truncate=False)


+----+-------------+--------+
|year|language_name|mentions|
+----+-------------+--------+
|2010|java         |52      |
|2010|php          |46      |
|2010|javascript   |44      |
|2010|python       |26      |
|2010|objective-c  |23      |
|2010|c            |20      |
|2010|ruby         |12      |
|2010|delphi       |8       |
|2010|r            |3       |
|2010|applescript  |3       |
|2010|perl         |3       |
|2010|bash         |3       |
|2010|haskell      |2       |
|2010|sed          |2       |
|2010|f#           |2       |
|2010|xpath        |1       |
|2010|racket       |1       |
|2010|actionscript |1       |
|2010|mouse        |1       |
|2010|scheme       |1       |
+----+-------------+--------+
only showing top 20 rows




## 8. Построение топ-10 по каждому году

Считаем ранг внутри каждого года по убыванию количества упоминаний.


In [63]:

ranking_window = Window.partitionBy("year").orderBy(F.desc("mentions"), F.asc("language_name"))

top_languages_long = (
    language_mentions
    .withColumn("rank", F.row_number().over(ranking_window))
    .where(F.col("rank") <= 10)
    .select("year", "rank", "language_name", "mentions")
    .orderBy("year", "rank")
)

top_languages_long.orderBy("year", "rank").show(200, truncate=False)


+----+----+-------------+--------+
|year|rank|language_name|mentions|
+----+----+-------------+--------+
|2010|1   |java         |52      |
|2010|2   |php          |46      |
|2010|3   |javascript   |44      |
|2010|4   |python       |26      |
|2010|5   |objective-c  |23      |
|2010|6   |c            |20      |
|2010|7   |ruby         |12      |
|2010|8   |delphi       |8       |
|2010|9   |applescript  |3       |
|2010|10  |bash         |3       |
|2011|1   |php          |102     |
|2011|2   |java         |93      |
|2011|3   |javascript   |83      |
|2011|4   |python       |37      |
|2011|5   |objective-c  |34      |
|2011|6   |c            |24      |
|2011|7   |ruby         |20      |
|2011|8   |perl         |9       |
|2011|9   |delphi       |8       |
|2011|10  |bash         |7       |
|2012|1   |php          |154     |
|2012|2   |javascript   |132     |
|2012|3   |java         |124     |
|2012|4   |python       |69      |
|2012|5   |objective-c  |45      |
|2012|6   |c        


## 9. Удобное представление для просмотра

Для отображения в ноутбуке соберём данные ещё и в широкую таблицу: одна строка — один год, колонки `top_1 ... top_10`.


In [65]:

wide_report = (
    top_languages_long
    .groupBy("year")
    .pivot("rank", list(range(1, 11)))
    .agg(F.first("language_name"))
)

for idx in range(1, 11):
    wide_report = wide_report.withColumnRenamed(str(idx), f"top_{idx}")

wide_report = wide_report.orderBy("year")
wide_report.show(truncate=False)


+----+----------+----------+----------+------+-----------+-----------+-----------+-----------+-----------+------+
|year|top_1     |top_2     |top_3     |top_4 |top_5      |top_6      |top_7      |top_8      |top_9      |top_10|
+----+----------+----------+----------+------+-----------+-----------+-----------+-----------+-----------+------+
|2010|java      |php       |javascript|python|objective-c|c          |ruby       |delphi     |applescript|bash  |
|2011|php       |java      |javascript|python|objective-c|c          |ruby       |perl       |delphi     |bash  |
|2012|php       |javascript|java      |python|objective-c|c          |ruby       |bash       |r          |lua   |
|2013|javascript|php       |java      |python|objective-c|c          |ruby       |r          |bash       |scala |
|2014|javascript|java      |php       |python|c          |objective-c|r          |ruby       |bash       |matlab|
|2015|javascript|java      |php       |python|r          |c          |objective-c|ruby  


## 10. Сохранение результата в Parquet

Основной результат сохраняем в **длинном формате** с колонками:
- `year`
- `rank`
- `language_name`
- `mentions`

Дополнительно данные партиционируются по `year`.


In [66]:

(top_languages_long
    .write
    .mode("overwrite")
    .partitionBy("year")
    .parquet(PARQUET_OUTPUT))

print("Результат сохранён в:", PARQUET_OUTPUT)


Результат сохранён в: /content/top_languages_2010_2020_parquet



## 11. Проверка сохранённого результата


In [67]:
restored = spark.read.parquet(PARQUET_OUTPUT)
restored.orderBy("year", "rank").show(200, truncate=False)
print("Количество строк в restored:", restored.count())
restored.select("year").distinct().orderBy("year").show(20, False)


+----+-------------+--------+----+
|rank|language_name|mentions|year|
+----+-------------+--------+----+
|1   |java         |52      |2010|
|2   |php          |46      |2010|
|3   |javascript   |44      |2010|
|4   |python       |26      |2010|
|5   |objective-c  |23      |2010|
|6   |c            |20      |2010|
|7   |ruby         |12      |2010|
|8   |delphi       |8       |2010|
|9   |applescript  |3       |2010|
|10  |bash         |3       |2010|
|1   |php          |102     |2011|
|2   |java         |93      |2011|
|3   |javascript   |83      |2011|
|4   |python       |37      |2011|
|5   |objective-c  |34      |2011|
|6   |c            |24      |2011|
|7   |ruby         |20      |2011|
|8   |perl         |9       |2011|
|9   |delphi       |8       |2011|
|10  |bash         |7       |2011|
|1   |php          |154     |2012|
|2   |javascript   |132     |2012|
|3   |java         |124     |2012|
|4   |python       |69      |2012|
|5   |objective-c  |45      |2012|
|6   |c            |

In [68]:
posts_with_year_and_tags.groupBy("year").count().orderBy("year").show(20, False)
top_languages_long.select("year").distinct().orderBy("year").show(20, False)

+----+-----+
|year|count|
+----+-----+
|2010|712  |
|2011|1172 |
|2012|1677 |
|2013|2082 |
|2014|2112 |
|2015|2172 |
|2016|2190 |
|2017|2076 |
|2018|2026 |
|2019|1423 |
+----+-----+

+----+
|year|
+----+
|2010|
|2011|
|2012|
|2013|
|2014|
|2015|
|2016|
|2017|
|2018|
|2019|
+----+




## 12. Остановка SparkSession


In [69]:

spark.stop()


## 13. Скачиваю отчёт, чтобы приложить к работе

In [70]:
from google.colab import files
import shutil

shutil.make_archive(
    "/content/top_languages_2010_2020_parquet",
    "zip",
    "/content/top_languages_2010_2020_parquet"
)

files.download("/content/top_languages_2010_2020_parquet.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>